# 05 GAFF2 Parameterisation

This notebook focuses only on **Stage 1: GAFF2 parameterisation with AmberTools**.

In simple terms, parameterisation converts a molecule file such as SDF into files that describe the physical model used for molecular dynamics. The most important outputs for the next notebook are:

- `.prmtop`: AMBER topology and force-field parameters
- `.inpcrd`: AMBER starting coordinates

This v2 workflow keeps the same general idea as iPHASimulator v1: prepare parameters first, then run simulation separately. The implementation here is deliberately simpler and specific to the v2 RDKit-built PHA oligomers.

## What AmberTools does here

The helper function `parameterize_gaff2` runs three AmberTools steps:

1. `antechamber`: assigns GAFF2 atom types and partial charges
2. `parmchk2`: looks for missing force-field terms and writes an `.frcmod` file
3. `tleap`: assembles AMBER `.prmtop` and `.inpcrd` files

For careful production work, the charge model matters. The default is AM1-BCC (`charge_method="bcc"`), which is more appropriate but can be slow. For quick debugging, `charge_method="gas"` avoids AM1-BCC and runs faster.

## Step 1: Check AmberTools

This cell does not run parameterisation. It only checks whether `antechamber`, `parmchk2`, and `tleap` are available on your `PATH`.

In [1]:
from iphasimulator.parameterization.gaff2 import ambertools_available

ambertools_available()

True

If the result is `False`, install the MD dependencies in your conda environment before running GAFF2:

```bash
conda install -c conda-forge ambertools openmm parmed mdtraj -y
```

Then restart Jupyter from the same environment.

## Step 2: Choose the SDF input

Run notebook `04_export_structures.ipynb` first if these SDF files do not exist. The default target below is `PHB4_R` because it is small and fast compared with longer-side-chain systems.

In [2]:
from pathlib import Path

repo_root = Path.cwd() if (Path.cwd() / "src" / "iphasimulator").exists() else Path.cwd().parent
output_root = repo_root / "examples" / "output"

target_name = "PHB4_R"
sdf_path = output_root / f"{target_name}.sdf"
gaff2_output_dir = output_root / "md_tests" / target_name.replace("_R", "") / "gaff2"

{
    "sdf_path": sdf_path,
    "sdf_exists": sdf_path.exists(),
    "gaff2_output_dir": gaff2_output_dir,
}

{'sdf_path': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/PHB4_R.sdf'),
 'sdf_exists': True,
 'gaff2_output_dir': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4/gaff2')}

## Step 3: Run GAFF2 parameterisation when ready

The next cell is deliberately disabled by default. Set `RUN_GAFF2 = True` only after:

- `sdf_exists` is `True`
- `ambertools_available()` is `True`
- you are comfortable with the run taking some time

For first debugging, keep `charge_method = "gas"`. For more realistic charge assignment, use `charge_method = "bcc"`.

In [5]:
from iphasimulator.parameterization.gaff2 import parameterize_gaff2

RUN_GAFF2 = False
charge_method = "gas"  # use "bcc" for AM1-BCC charges

if RUN_GAFF2:
    gaff2_outputs = parameterize_gaff2(
        sdf_path,
        gaff2_output_dir,
        name=target_name.replace("_R", ""),
        net_charge=0,
        residue_name="PHA",
        charge_method=charge_method,
        verbose=True,
    )
    gaff2_outputs
else:
    "Set RUN_GAFF2 = True to run AmberTools parameterisation."

## Step 4: Check expected outputs

After a successful run, `prmtop_exists` and `inpcrd_exists` should be `True`. These two files are the inputs for notebook `06_openmm_setup.ipynb`.

In [6]:
amber_name = target_name.replace("_R", "")
prmtop_path = gaff2_output_dir / f"{amber_name}.prmtop"
inpcrd_path = gaff2_output_dir / f"{amber_name}.inpcrd"

{
    "prmtop_path": prmtop_path,
    "prmtop_exists": prmtop_path.exists(),
    "inpcrd_path": inpcrd_path,
    "inpcrd_exists": inpcrd_path.exists(),
    "timing_log": gaff2_output_dir / "timing.log",
    "antechamber_log": gaff2_output_dir / "antechamber.log",
}

{'prmtop_path': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4/gaff2/PHB4.prmtop'),
 'prmtop_exists': True,
 'inpcrd_path': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4/gaff2/PHB4.inpcrd'),
 'inpcrd_exists': True,
 'timing_log': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4/gaff2/timing.log'),
 'antechamber_log': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4/gaff2/antechamber.log')}

## Command-line equivalent

For repeatable runs, use the script from the terminal:

```bash
PYTHONPATH=src python examples/run_gaff2_openmm_test.py --target PHB4 --skip-openmm --skip-am1-bcc
```

Remove `--skip-am1-bcc` when you want AM1-BCC charge generation.